# 워밍업 — 들어오는 게 줄었는데 왜 계속 늘까

웹 챕터(워밍업)와 **같은 예제, 같은 수치**를 파이썬으로 직접 계산해 봅니다.
웹의 실험실(BathtubLab)과 같은 월별 데이터를 쓰므로 결과가 정확히 일치해요.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('sample')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 무대: 월별 가입(유입)과 이탈(유출)

가상 회사의 사내 AI 도구, 첫 1년입니다(처음 사용자 0명). 매달 사용자 수는 **가입 − 이탈**만큼 바뀝니다.

$$\text{이번 달 말 사용자 수} = \text{지난달 말 사용자 수} + \text{가입} - \text{이탈}$$

In [ ]:
inflow  = [20, 60, 100, 140, 160, 150, 120, 90, 60, 40, 30, 20]   # 신규 가입 = 유입 (명/월)
outflow = [0, 5, 10, 20, 30, 45, 60, 70, 75, 75, 70, 65]          # 이탈 = 유출 (명/월)

users = []        # 월말 사용자 수 = 저량
s = 0
for i, o in zip(inflow, outflow):
    s += i - o    # 저량은 순유량(유입 − 유출)이 쌓인 것
    users.append(s)

for m, (i, o, u) in enumerate(zip(inflow, outflow, users), start=1):
    print(f'{m:2d}월  가입 {i:3d}  이탈 {o:3d}  순유량 {i - o:+4d}  →  월말 사용자 {u:3d}')

peak_in = inflow.index(max(inflow)) + 1
peak_users = users.index(max(users)) + 1
print(f'가입 정점: {peak_in}월 ({max(inflow)}명/월),  사용자 정점: {peak_users}월 ({max(users)}명)')

# 웹 실험실(BathtubLab)과 같은 수치인지 자가검증
assert users == [20, 75, 165, 285, 415, 520, 580, 600, 585, 550, 510, 465]
assert peak_in == 5 and peak_users == 8 and max(users) == 600
assert users[2] == 165   # 3월 말 (연습 문제)
print('✅ 웹과 일치: 가입 정점 5월, 사용자 정점 8월(600명)')

## 2. 왜 8월인가: 정점은 유입 = 유출인 곳

순유량(가입 − 이탈)의 **부호**가 저량의 방향을 정합니다. +인 동안 늘고, −가 되면 줄어요.
그래서 정점은 순유량이 +에서 −로 바뀌는 곳, 곧 가입 곡선과 이탈 곡선이 만나는 곳에 생깁니다.
가입이 줄어드는 6~8월에도 사용자 수가 늘었던 이유예요.

In [ ]:
net = [i - o for i, o in zip(inflow, outflow)]
last_up = max(m for m, n in enumerate(net, start=1) if n > 0)
first_down = min(m for m, n in enumerate(net, start=1) if n < 0)
print('월별 순유량:', net)
print(f'순유량이 +인 마지막 달: {last_up}월, 처음으로 −가 되는 달: {first_down}월')

assert (last_up, first_down) == (8, 9)
assert net[5:8] == [105, 60, 20]     # 6~8월: 가입은 줄지만 순유량은 여전히 +
assert net[8] == -15                  # 9월: 처음으로 −
print('✅ 정점(8월) = 순유량이 +에서 −로 바뀌는 곳')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

months = list(range(1, 13))
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6.2), sharex=True)
ax1.plot(months, inflow, marker='o', color='#6366f1', label='신규 가입(유입)')
ax1.plot(months, outflow, marker='o', color='#f43f5e', label='이탈(유출)')
ax1.axvline(5, ls=':', color='#6366f1')
ax1.set_ylabel('명/월')
ax1.set_title('유량: 가입은 5월에 꺾였다')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot([0] + months, [0] + users, marker='o', color='#059669', lw=2.5, label='사용자 수(저량)')
ax2.axvline(8, ls=':', color='#059669')
ax2.annotate('정점 8월, 600명', (8, 600), textcoords='offset points', xytext=(8, -18), color='#059669')
ax2.set_xlabel('월')
ax2.set_ylabel('명')
ax2.set_title('저량: 사용자 수는 8월까지 계속 늘었다')
ax2.set_xticks([0] + months)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 보너스: 12월 값이 계속된다면

12월 말 사용자는 465명, 12월 가입 20명·이탈 65명입니다. 이 값이 그대로 계속된다고 **가정**하면
매달 45명씩 줄어 몇 달 뒤 0이 될까요?

In [ ]:
months_to_zero = users[-1] / (outflow[-1] - inflow[-1])
print(f'{users[-1]} ÷ {outflow[-1] - inflow[-1]} = {months_to_zero:.2f}개월')
assert round(months_to_zero, 1) == 10.3

# 그런데 실제로는 이탈이 '지금 사용자 중 일부'라서 사용자 수가 줄면 이탈도 준다.
# 예: 매달 사용자의 14%가 이탈한다면(12월 65 ÷ 465 ≈ 14%), 사용자는 0이 되지 않고 천천히 줄어든다.
rate = outflow[-1] / users[-1]
u = users[-1]
for _ in range(12):
    u += inflow[-1] - rate * u
print(f'이탈을 사용자 수의 {rate*100:.0f}%로 두면 1년 뒤 사용자: {u:.0f}명 (0이 아니라 {inflow[-1]/rate:.0f}명 쪽으로 다가감)')
assert round(u) == 196
print('✅ 저량이 다시 유량(이탈)을 정하는 고리 = 피드백 루프, 1장부터')

## 4. 오늘의 발견

- **저량(stock)** 은 순유량(유입 − 유출)이 쌓인 것: 월말 사용자 수 = 지난달 + 가입 − 이탈.
- 저량의 방향은 유입의 오르내림이 아니라 **순유량의 부호**가 정한다. 가입 정점(5월)과 사용자 정점(8월)은 석 달 차이.
- 정점은 **유입 = 유출**인 곳(두 곡선이 만나는 곳)에 생긴다.
- 실제 이탈은 사용자 수를 보고 정해진다(저량 → 유량). 이 고리가 **피드백 루프**이고 1장의 주제다.

다음 장(1장 큰 그림)에서 시간의 모양을 만드는 네 부품을 하나씩 켜고 끕니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**